
# Qwen2.5-3B Abstract Evaluator SFT (Unsloth QLoRA)

This notebook fine-tunes `Qwen/Qwen2.5-3B-Instruct` for abstract-quality scoring + rationale generation using chat-format SFT data.

Target mapping:

- **Input**: `Task + Reference + Rubric + Submission`
- **Output**: JSON string with `score` and `rationale`

Designed for **RTX 4060 8GB VRAM** with memory-efficient Unsloth QLoRA.


In [ ]:

## If needed, uncomment and run once in your environment.
!pip install -U "unsloth[colab-new]" datasets transformers trl accelerate evaluate rouge_score sacrebleu bert-score wandb scikit-learn pandas numpy


In [ ]:

import os
import re
import ast
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OPENREVIEW_CSV = DATA_DIR / "openreview" / "iclr2024_openreview_7000.csv"

# Preferred final split files (chat-format JSONL with messages field)
TRAIN_JSONL = DATA_DIR / "train.jsonl"
VAL_JSONL   = DATA_DIR / "dev.jsonl"
TEST_JSONL  = DATA_DIR / "test.jsonl"

# Fallback processed path if we auto-build from CSV
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Using train/dev/test JSONL if present:", TRAIN_JSONL.exists(), VAL_JSONL.exists(), TEST_JSONL.exists())
print("Fallback OpenReview CSV exists:", OPENREVIEW_CSV.exists())



## Dataset Utilities

This block supports two modes:

1. **Preferred**: use existing `train.jsonl / dev.jsonl / test.jsonl` with `messages`.
2. **Fallback**: build a training set from OpenReview CSV with simple weak labels, then split.

If you already generated your high-quality synthetic dataset, place the JSONL files in `data/` and skip fallback quality concerns.


In [ ]:

TASK_TEXT = "Evaluate the quality of the following research abstract for conference acceptance."
REFERENCE_TEXT = "A strong research abstract clearly presents the problem, methodology, contribution, and experimental evidence."
RUBRIC_TEXT = """0 = Strong reject
1 = Reject
2 = Borderline
3 = Accept
4 = Strong accept"""


def normalize_score(decision: str) -> int:
    if not isinstance(decision, str):
        return 2
    d = decision.lower()
    if "strong accept" in d:
        return 4
    if "accept" in d:
        return 3
    if "border" in d:
        return 2
    if "strong reject" in d:
        return 0
    if "reject" in d:
        return 1
    return 2


def synthesize_rationale(abstract: str, score: int) -> str:
    length = len(str(abstract).split())
    if score <= 1:
        return "The abstract lacks sufficient methodological and empirical detail, making the contribution difficult to verify for conference standards."
    if score == 2:
        return "The abstract presents a relevant topic, but important details about methodology or concrete evidence are limited, resulting in a borderline evaluation."
    if score == 3:
        return "The abstract clearly states the problem and approach with generally convincing evidence, though some details could be more specific for a stronger case."
    return "The abstract clearly defines the problem, method, contribution, and empirical support, providing a strong case for conference acceptance."


def build_user_prompt(submission: str) -> str:
    return (
        f"Task:{TASK_TEXT}"
        f"Reference:{REFERENCE_TEXT}"
        f"Rubric:{RUBRIC_TEXT}"
        f"Submission:{submission}"
    )


def build_assistant_json(score: int, rationale: str) -> str:
    return json.dumps({"score": int(score), "rationale": str(rationale)}, ensure_ascii=False)


def row_to_messages(submission: str, score: int, rationale: str):
    return [
        {"role": "user", "content": build_user_prompt(submission)},
        {"role": "assistant", "content": build_assistant_json(score, rationale)},
    ]


def load_or_build_splits() -> DatasetDict:
    if TRAIN_JSONL.exists() and VAL_JSONL.exists() and TEST_JSONL.exists():
        dsd = DatasetDict({
            "train": load_dataset("json", data_files=str(TRAIN_JSONL), split="train"),
            "validation": load_dataset("json", data_files=str(VAL_JSONL), split="train"),
            "test": load_dataset("json", data_files=str(TEST_JSONL), split="train"),
        })
        return dsd

    if not OPENREVIEW_CSV.exists():
        raise FileNotFoundError("No train/dev/test JSONL found and fallback CSV missing.")

    df = pd.read_csv(OPENREVIEW_CSV)
    df = df.dropna(subset=["abstract"]).copy()

    df["score"] = df["decision"].apply(normalize_score)
    df["rationale"] = [synthesize_rationale(a, s) for a, s in zip(df["abstract"], df["score"])]
    df["messages"] = [row_to_messages(a, s, r) for a, s, r in zip(df["abstract"], df["score"], df["rationale"])]

    keep_cols = [c for c in ["paper_id", "title", "abstract", "messages", "score", "rationale"] if c in df.columns]
    df = df[keep_cols].copy()

    n_train, n_val, n_test = 8000, 1000, 1000
    needed = n_train + n_val + n_test
    if len(df) < needed:
        raise ValueError(f"Need at least {needed} rows, found {len(df)}. Provide prepared JSONL splits.")

    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    train_df = df.iloc[:n_train]
    val_df = df.iloc[n_train:n_train+n_val]
    test_df = df.iloc[n_train+n_val:n_train+n_val+n_test]

    for split_name, split_df, out_path in [
        ("train", train_df, TRAIN_JSONL),
        ("validation", val_df, VAL_JSONL),
        ("test", test_df, TEST_JSONL),
    ]:
        records = split_df[["messages"]].to_dict(orient="records")
        with open(out_path, "w", encoding="utf-8") as f:
            for rec in records:
                f.write(json.dumps(rec, ensure_ascii=False) + "")

    dsd = DatasetDict({
        "train": Dataset.from_pandas(train_df[["messages"]].reset_index(drop=True), preserve_index=False),
        "validation": Dataset.from_pandas(val_df[["messages"]].reset_index(drop=True), preserve_index=False),
        "test": Dataset.from_pandas(test_df[["messages"]].reset_index(drop=True), preserve_index=False),
    })
    return dsd


datasets_dict = load_or_build_splits()
print(datasets_dict)
print("Train sample:")
print(datasets_dict["train"][0]["messages"][0]["content"][:300], "...")
print(datasets_dict["train"][0]["messages"][1]["content"])



## Load Model (Unsloth QLoRA)

Why no manual 50-70% freezing?

- QLoRA already freezes base weights and trains only LoRA adapters.
- This is more memory-efficient and typically better than manually freezing large contiguous layer blocks for this setup.
- We control trainable capacity via LoRA rank/alpha/target modules.


In [ ]:

import torch
from unsloth import FastLanguageModel

max_seq_length = 1024
load_in_4bit = True

dtype = None
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    dtype = torch.bfloat16 if major >= 8 else torch.float16
else:
    dtype = torch.float32

model_name = "Qwen/Qwen2.5-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print("Tokenizer pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)


In [ ]:

def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = datasets_dict["train"].map(apply_chat_template)
val_ds = datasets_dict["validation"].map(apply_chat_template)
test_ds = datasets_dict["test"].map(apply_chat_template)

print(train_ds[0]["text"][:500])


## Weights & Biases

In [ ]:

import wandb

WANDB_PROJECT = os.getenv("WANDB_PROJECT", "abstract-evaluator-qwen25-3b")
WANDB_RUN_NAME = os.getenv("WANDB_RUN_NAME", "qwen25-3b-unsloth-qlora")

wandb.login()
wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME)


## Train (SFTTrainer)

In [ ]:

from trl import SFTTrainer
from transformers import TrainingArguments

num_epochs = 3

training_args = TrainingArguments(
    output_dir=str(PROJECT_ROOT / "outputs" / "qwen25_3b_abstract_eval"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    weight_decay=0.01,
    num_train_epochs=num_epochs,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=(dtype == torch.float16),
    bf16=(dtype == torch.bfloat16),
    report_to=["wandb"],
    run_name=WANDB_RUN_NAME,
    gradient_checkpointing=True,
    dataloader_pin_memory=True,
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=training_args,
)

train_result = trainer.train()
print(train_result)


In [ ]:

adapter_dir = PROJECT_ROOT / "outputs" / "qwen25_3b_abstract_eval" / "lora_adapter"
trainer.model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Saved adapter to:", adapter_dir)


## Validation + Test Generation and Metrics

In [ ]:

import evaluate
from tqdm.auto import tqdm

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")

FastLanguageModel.for_inference(model)


def parse_gold(messages):
    text = messages[1]["content"]
    try:
        obj = json.loads(text)
    except Exception:
        obj = {"score": None, "rationale": text}
    return obj


def parse_pred(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return {"score": None, "rationale": text.strip()}
    chunk = match.group(0)
    try:
        return json.loads(chunk)
    except Exception:
        return {"score": None, "rationale": text.strip()}


def generate_predictions(ds, max_new_tokens=180):
    preds, refs = [], []
    pred_scores, ref_scores = [], []

    for row in tqdm(ds):
        msgs = row["messages"]
        prompt = tokenizer.apply_chat_template(
            [msgs[0]], tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                top_p=1.0,
                eos_token_id=tokenizer.eos_token_id,
            )
        gen_text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        pred_obj = parse_pred(gen_text)
        gold_obj = parse_gold(msgs)

        preds.append(str(pred_obj.get("rationale", "")))
        refs.append(str(gold_obj.get("rationale", "")))
        pred_scores.append(pred_obj.get("score", None))
        ref_scores.append(gold_obj.get("score", None))

    return preds, refs, pred_scores, ref_scores


val_preds, val_refs, val_pred_scores, val_ref_scores = generate_predictions(datasets_dict["validation"])
test_preds, test_refs, test_pred_scores, test_ref_scores = generate_predictions(datasets_dict["test"])

val_rouge = rouge_metric.compute(predictions=val_preds, references=val_refs)
val_bleu = bleu_metric.compute(predictions=val_preds, references=[[r] for r in val_refs])
val_bertscore = bertscore_metric.compute(predictions=val_preds, references=val_refs, lang="en")

print("Validation ROUGE:", val_rouge)
print("Validation BLEU:", val_bleu)
print("Validation BERTScore F1 mean:", float(np.mean(val_bertscore["f1"])))


In [ ]:

test_rouge = rouge_metric.compute(predictions=test_preds, references=test_refs)
test_bleu = bleu_metric.compute(predictions=test_preds, references=[[r] for r in test_refs])
test_bertscore = bertscore_metric.compute(predictions=test_preds, references=test_refs, lang="en")

valid_pairs = [(p, r) for p, r in zip(test_pred_scores, test_ref_scores) if isinstance(p, int) and isinstance(r, int)]
score_acc = float(np.mean([int(p == r) for p, r in valid_pairs])) if valid_pairs else None

results = {
    "test_rouge": test_rouge,
    "test_bleu": test_bleu,
    "test_bertscore_f1_mean": float(np.mean(test_bertscore["f1"])),
    "test_score_accuracy": score_acc,
    "num_valid_score_pairs": len(valid_pairs),
}

print(json.dumps(results, indent=2))

wandb.log({
    "test/rouge1": test_rouge.get("rouge1", 0.0),
    "test/rouge2": test_rouge.get("rouge2", 0.0),
    "test/rougeL": test_rouge.get("rougeL", 0.0),
    "test/bleu": test_bleu.get("bleu", 0.0),
    "test/bertscore_f1_mean": float(np.mean(test_bertscore["f1"])),
    "test/score_accuracy": score_acc if score_acc is not None else 0.0,
})

results_path = PROJECT_ROOT / "outputs" / "qwen25_3b_abstract_eval" / "final_metrics.json"
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("Saved:", results_path)



## Notes

- For 8GB VRAM, start with `max_seq_length=1024`, `batch_size=1`, `grad_accum=16`, QLoRA 4-bit.
- If OOM occurs:
  - reduce `max_seq_length` to `768` or `512`
  - increase `gradient_accumulation_steps` instead of batch size
  - set LoRA `r=8`
- If underfitting, try 4 epochs; if validation loss rises while train loss drops, keep 2-3 epochs.
